# AgentCore Memory를 사용하는 LlamaIndex - 의료 지식 도우미(장기 메모리)

## 소개

이 Notebook에서는 Amazon Bedrock AgentCore Memory 기능을 LlamaIndex와 통합하여 여러 환자 상담과 의료 사례에 걸쳐 **장기 메모리**를 유지하는 의료 지식 도우미를 만드는 방법을 살펴봅니다. 이를 통해 도우미는 수개월 또는 수년에 걸쳐 의료 지식을 축적하고 환자 진료를 추적할 수 있습니다.

## 아키텍처 개요

![LlamaIndex AgentCore 장기 메모리 아키텍처](LlamaIndex-AgentCore-LTM-Arch.png)

## 튜토리얼 세부 정보

**튜토리얼 세부 정보:**
- **튜토리얼 유형**: Session 간 장기 메모리
- **Agent 사용 사례**: 의료 지식 도우미
- **Agentic Framework**: LlamaIndex
- **LLM model**: Anthropic Claude 3.7 Sonnet
- **튜토리얼 구성 요소**: AgentCore 장기 메모리, LlamaIndex Agent, 의료 Tool
- **예제 난이도**: 고급

## 비즈니스 가치

**Enterprise Medical Intelligence**: 환자 지식을 축적하고 치료 변화 과정을 추적하며 여러 사례와 기간에 걸쳐 종합적인 의료 기록을 유지하는 지속형 AI 메모리로 의료 업무를 혁신합니다.

**주요 전문적 이점:**
- **환자 진료 연속성**: 진료 과정과 의료진 간에 지식을 원활하게 이전
- **임상 메모리**: 중요한 환자 기록, 치료, 결과를 영구 보존
- **환자 간 Intelligence**: 여러 환자 사례에서 pattern과 연관성 식별
- **우수한 치료**: 과거 환자 데이터를 활용하여 더 나은 진료 결정
- **인구 건강**: 장기 환자 진료의 상세 맥락 유지
- **품질 개선**: 치료 protocol과 시간에 따른 효과 추적

## 장기 메모리 구성

**기술 설정**: 이 튜토리얼에서는 Semantic Strategy가 적용된 AgentCore Memory를 사용하여 데이터를 12개월간 보존합니다.
- **Memory 유형**: Insight를 자동으로 추출하는 semantic strategy
- **보존 기간**: 환자 진료 연속성을 위한 365일 event 만료 기간
- **Session 간 구성**: 동일한 actor_id + memory_id, 의료 업무 기간별로 서로 다른 session_id
- **검색 기능**: 전체 환자 기록을 semantic search하는 기본 제공 memory 검색 tool

## 기술 개요

**주요 장기 메모리 구성 요소:**
1. **Semantic Strategy 구성**: SemanticStrategy를 사용하여 insight를 자동 추출하고 365일간 보존
2. **Session 간 지속성**: 동일한 actor_id + memory_id와 기간별로 다른 session_id를 사용하여 지식 연속성 구현
3. **Custom Memory 검색 Tool**: AgentCore 기본 search_long_term_memories()를 LlamaIndex FunctionTool로 wrapping
4. **Semantic 처리 Pipeline**: 대화 event를 semantic memory로 변환하기 위해 120초 대기
5. **동적 Session 관리**: 유연한 session 처리를 위해 memory.context.session_id 사용

**다음 내용을 학습합니다:**

- 여러 환자 상담에 걸쳐 지속되는 AgentCore Memory 생성
- 시간에 따라 의료 지식 축적
- 환자 기록과 치료 결과를 대상으로 semantic search 구현
- 치료 효과와 의료 insight 변화 추적
- Session 간 의료 지식 지속성 및 검색 테스트

## 시나리오 배경

이 예제에서는 수개월과 수년에 걸친 여러 환자 상담에서 의료 지식을 유지하는 "Medical Knowledge Assistant"를 만듭니다. 도우미는 AgentCore Memory를 사용하여 환자 기록, 치료 protocol, 약물 상호 작용, 임상 결과에 관한 지속형 지식 base를 구축합니다. 이 지식은 시간에 따라 축적되고 발전하여 정교한 장기 의료 분석을 지원합니다.

## 사전 요구 사항

- Python 3.10+
- 적절한 권한이 있는 AWS account
- AgentCore Memory 권한이 있는 AWS IAM role:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock model에 대한 액세스

## 1단계: Dependency 설치 및 설정

In [ ]:
# Semantic strategy toolkit을 포함한 필수 library 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3 bedrock-agentcore-starter-toolkit

In [ ]:
# 필요한 component import
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies.semantic import (
    SemanticStrategy,
)
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

print("✅ All dependencies imported successfully!")

## 2단계: AgentCore Memory 구성

장기 의료 지식을 위한 AgentCore Memory resource를 생성하거나 가져옵니다.

In [ ]:
# 장기 지속성을 위해 Semantic Strategy가 적용된 AgentCore Memory 생성
region = os.getenv("AWS_REGION", "us-east-1")
memory_manager = MemoryManager(region_name=region)

try:
    # Insight 자동 추출용 semantic strategy로 memory 생성
    memory = memory_manager.get_or_create_memory(
        name=f"MedicalAssistantSemantic_{int(datetime.now().timestamp())}",
        strategies=[SemanticStrategy(name="medicalLongTermMemory")],
        event_expiry_days=365,  # 의료 record를 12개월간 보존
    )
    memory_id = memory.get("id")
    print(f"✅ Created Semantic Memory: {memory_id}")
    print(f"   Status: {memory.get('status')}")
    print(f"   Strategies: {[s.get('name') if isinstance(s, dict) else str(s) for s in memory.get('strategies', [])]}")

    # Memory가 ACTIVE 상태가 될 때까지 대기
    if memory.get("status") != "ACTIVE":
        print(f"\n⏳ Waiting for memory to become ACTIVE (currently {memory.get('status')})...")
        import time

        max_wait = 300  # 최대 5분
        waited = 0
        while waited < max_wait:
            time.sleep(10)
            waited += 10
            # 상태 확인
            current_memory = memory_manager.get_memory(memory_id)
            status = current_memory.get("status")
            print(f"   [{waited}s] Status: {status}")
            if status == "ACTIVE":
                print(f"✅ Memory is now ACTIVE! (took {waited} seconds)")
                break
        else:
            print(f"⚠️  Memory still not ACTIVE after {max_wait}s. Proceeding anyway...")

except Exception as e:
    print(f"❌ Error creating memory: {e}")
    memory_id = "your-memory-id-here"  # 기존 memory ID로 교체

## 3단계: 의료 Tool 구현

장기 의료 분석을 위한 전문 tool을 정의합니다.

In [ ]:
def assess_patient_symptoms(patient_id: str, symptoms: str, severity: str, duration: str) -> str:
    """Assess patient symptoms with severity and duration tracking"""
    return f"🩺 Assessed symptoms for {patient_id} (Severity: {severity}, Duration: {duration})"


def check_drug_interactions(patient_id: str, medications: str, interaction_level: str, recommendations: str) -> str:
    """Check drug interactions with safety recommendations"""
    return f"💊 {patient_id} drug interaction check: {interaction_level} - {recommendations}"


def document_treatment_protocol(protocol_type: str, indication: str, effectiveness: str, side_effects: str) -> str:
    """Document treatment protocol with effectiveness and side effects"""
    print(f"📋 Treatment protocol: {protocol_type} for {indication} (Effectiveness: {effectiveness})")
    return f"Documented treatment protocol: {protocol_type}"


def update_clinical_guideline(patient_id: str, guideline_type: str, recommendation: str, evidence_level: str) -> str:
    """Update clinical guideline for specific patient"""
    print(f"📖 Clinical guideline: {patient_id} - {guideline_type} ({evidence_level} evidence)")
    return f"Updated guideline for {patient_id}"


def log_treatment_outcome(patient_id: str, treatment: str, outcome: str, follow_up_needed: str) -> str:
    """Log treatment outcome with follow-up requirements"""
    print(f"🏥 Treatment outcome: {patient_id} - {treatment}: {outcome}")
    return f"Logged outcome for {patient_id}"


def log_medical_milestone(quarter: str, milestone: str, details: str) -> str:
    """Log a medical milestone with quarter and detailed progress"""
    print(f"🎯 {quarter} milestone: {milestone}")
    return f"Logged milestone for {quarter}: {milestone} - {details}"


def track_clinical_metrics(metric_type: str, value: str, patient_id: str, quarter: str) -> str:
    """Track specific clinical metrics with patient and timeline"""
    print(f"📊 {quarter}: {metric_type} = {value} (for {patient_id})")
    return f"Tracked {metric_type}: {value} for {patient_id} in {quarter}"


def save_medical_insight(insight: str, quarter: str, clinical_context: str) -> str:
    """Save medical insights with clinical context"""
    print(f"💡 {quarter} insight: {insight[:50]}...")
    return f"Saved {quarter} insight with clinical context: {clinical_context}"


# Agent용 tool object 생성
medical_tools = [
    FunctionTool.from_defaults(fn=assess_patient_symptoms),
    FunctionTool.from_defaults(fn=check_drug_interactions),
    FunctionTool.from_defaults(fn=document_treatment_protocol),
    FunctionTool.from_defaults(fn=update_clinical_guideline),
    FunctionTool.from_defaults(fn=log_treatment_outcome),
    FunctionTool.from_defaults(fn=log_medical_milestone),
    FunctionTool.from_defaults(fn=track_clinical_metrics),
    FunctionTool.from_defaults(fn=save_medical_insight),
]

print("✅ Medical tools created!")

## 3b단계: Memory 검색 Tool 추가

Agent가 장기 메모리를 검색할 수 있는 tool을 생성합니다.

In [ ]:
def create_memory_retrieval_tool(memory_id: str, actor_id: str, region: str):
    """에이전트가 자체 장기 메모리를 검색하는 도구를 생성합니다."""

    def search_long_term_memory(query: str) -> str:
        """Search long-term memory for relevant medical information about patients, treatments, protocols, and outcomes.

        Use this tool when you need to recall:
        - Patient information (symptoms, treatments, outcomes)
        - Treatment protocols and their effectiveness
        - Drug interactions and safety profiles
        - Clinical guidelines and recommendations
        - Medical insights and lessons learned

        Args:
            query: Search query describing what information you need (e.g., 'PATIENT-001 symptoms', 'diabetes protocols', 'Q1 outcomes')

        Returns:
            Relevant information from long-term memory
        """
        try:
            from bedrock_agentcore.memory.session import MemorySessionManager

            # Session manager 생성 (memory_id와 region만 필요)
            session_manager = MemorySessionManager(memory_id=memory_id, region_name=region)

            # Semantic strategy namespace에서 장기 메모리 검색
            results = session_manager.search_long_term_memories(
                query=query,
                namespace_prefix="/strategies/",  # Semantic strategy namespace에서 검색
                top_k=5,
                max_results=10,
            )

            if not results:
                return "No relevant information found in long-term memory. This might be new information or the memory extraction may still be processing."

            # Agent용 결과 형식 지정
            output = "📚 Retrieved from long-term memory:\\n\\n"
            for i, result in enumerate(results, 1):
                # MemoryRecord object의 content attribute에 액세스
                content = getattr(result, "content", str(result))
                # 매우 긴 content 자르기
                if len(content) > 300:
                    content = content[:300] + "..."
                output += f"{i}. {content}\\n\\n"

            return output

        except Exception as e:
            return f"⚠️ Error searching memory: {str(e)}. Proceeding without historical context."

    return FunctionTool.from_defaults(fn=search_long_term_memory)


# Memory 검색 tool 생성
memory_search_tool = create_memory_retrieval_tool(memory_id, "medical-assistant", region)

# Tool 목록에 memory 검색 추가
medical_tools_with_memory = medical_tools + [memory_search_tool]

print(f"✅ Memory retrieval tool created! Total tools: {len(medical_tools_with_memory)}")
print("   Using namespace: /strategies/ (for semantic strategy compatibility)")

## 3c단계: Memory 구성 확인

Semantic strategy가 올바르게 구성되었는지 확인합니다.

In [ ]:
# Memory 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"Strategies: {memory_info.get('strategies')}")
print(f"Status: {memory_info.get('status')}")
print(f"Name: {memory_info.get('name')}")

# Strategy 세부 정보 표시
strategies = memory_info.get("strategies", [])
for strategy in strategies:
    print("\nStrategy Details:")
    print(f"  Name: {strategy.get('name')}")
    print(f"  Type: {strategy.get('type')}")
    print(f"  Status: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 3c단계: Memory 구성 확인

Semantic strategy가 올바르게 구성되었는지 확인합니다.

In [ ]:
# Memory 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"Strategies: {memory_info.get('strategies')}")
print(f"Status: {memory_info.get('status')}")
print(f"Name: {memory_info.get('name')}")

# Strategy 세부 정보 표시
strategies = memory_info.get("strategies", [])
for strategy in strategies:
    print("\nStrategy Details:")
    print(f"  Name: {strategy.get('name')}")
    print(f"  Type: {strategy.get('type')}")
    print(f"  Status: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 4단계: Multi-Session Agent 구현

서로 다른 의료 업무 기간을 시뮬레이션하는 helper function을 생성합니다.

In [ ]:
# 장기 메모리 구성 (session 간)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
ASSISTANT_ID = "medical-assistant"  # 모든 session에서 동일한 assistant


def create_medical_session(session_name: str):
    """장기 메모리가 유지되는 새 의료 세션을 생성합니다."""
    context = AgentCoreMemoryContext(
        actor_id=ASSISTANT_ID,  # 동일한 assistant
        memory_id=memory_id,  # 동일한 memory store (장기 메모리 활성화)
        session_id=f"medical-{session_name}",  # 기간별로 다른 session
        namespace="/medical-analysis/",
    )

    memory = AgentCoreMemory(context=context)
    llm = BedrockConverse(model=MODEL_ID)
    agent = FunctionAgent(
        tools=medical_tools_with_memory,  # Memory 검색 기능이 있는 tool 사용
        llm=llm,
        verbose=True,  # Memory 검색 시점을 확인하도록 verbose 활성화
        system_prompt="""You are a senior medical assistant with access to long-term memory.
        
CRITICAL: When asked about patients, treatments, protocols, or historical information, 
you MUST use the search_long_term_memory tool FIRST before responding.

For example:
- "What patients am I treating?" → Use search_long_term_memory("patients treatments")
- "What protocols have I used?" → Use search_long_term_memory("medical protocols")
- "What outcomes have I achieved?" → Use search_long_term_memory("patient outcomes")

Always provide conclusive, complete responses without asking follow-up questions.\n
Execute all requested actions immediately and completely. Provide detailed, professional responses.""",
    )

    return agent, memory


print("✅ Multi-session Medical Knowledge Assistant setup complete!")

## 5단계: Q1 의료 Session - 초기 환자 평가

첫 의료 session을 시작하고 환자 baseline을 설정합니다.

In [ ]:
# === Q1 의료 SESSION 시작 ===
print("🗓️ === Q1: INITIAL PATIENT ASSESSMENT ===")

agent_q1, memory_q1 = create_medical_session("q1")

# 초기 환자 증상 평가
response = await agent_q1.run(
    "I'm Senior Medical Assistant Dr. Maria Rodriguez. Assess patient symptoms for 'PATIENT-001' with symptoms 'fatigue, increased thirst, frequent urination, blurred vision', "
    "severity 'Moderate', duration '3 weeks'.",
    memory=memory_q1,
)

print("🎯 Q1 Initial Assessment:")
print(response)

In [ ]:
# 초기 임상 guideline 문서화
response = await agent_q1.run(
    "Update clinical guideline for 'PATIENT-001': guideline type 'Diabetes Management', "
    "recommendation 'initiate metformin 500mg BID, lifestyle counseling, glucose monitoring', evidence level 'high'.",
    memory=memory_q1,
)
print("💭 Q1 Clinical Guideline:", response)

response = await agent_q1.run(
    "Update clinical guideline for 'PATIENT-001': guideline type 'Lifestyle Modification', "
    "recommendation 'dietary consultation, exercise program, weight management target 10% reduction', evidence level 'high'.",
    memory=memory_q1,
)
print("💭 Q1 Lifestyle Guideline:", response)
# 의료 결과 명시적 추적
await agent_q1.run(
    "Save medical finding: finding 'Patient responds well to treatment protocol', confidence 'high'.",
    memory=memory_q1,
)

In [ ]:
# Semantic memory 처리 시간 확보
import asyncio

print("\n⏳ Waiting for medical memory extraction...")
await asyncio.sleep(120)
print("✅ Medical memory processing complete")

In [ ]:
# Event 저장 여부 확인
print("\n🔍 Verifying events were stored...")
try:
    client = MemoryClient(region_name=region)
    events = client.list_events(
        memory_id=memory_id,
        actor_id="medical-assistant",  # Domain별 ID로 교체됨
        session_id=memory_q1.context.session_id,  # 동적 session ID
    )
    print(f"✅ Stored {len(events)} conversational events in session")
except Exception as e:
    print(f"⚠️  Could not verify events: {e}")

# Semantic memory 처리 시간 확보
import asyncio

print("\n⏳ Waiting for semantic memory extraction and indexing...")
print("   (AgentCore processes conversational events in the background)")
await asyncio.sleep(120)  # Memory 추출 대기 시간 연장
print("✅ Memory processing complete - memories should now be searchable")

## 6단계: Q2 의료 Session - 치료 Protocol 업데이트

장기 메모리 검색을 테스트하고 치료 반응에 맞게 조정합니다.

In [ ]:
# === Q2 의료 SESSION 시작 ===
print("\n🗓️ === Q2: TREATMENT PROTOCOL UPDATE (NEW SESSION) ===")

agent_q2, memory_q2 = create_medical_session("q2")

# Session 간 환자 회상 테스트 - agent가 search_long_term_memory tool을 사용해야 함
print("\n🧠 Testing memory retrieval across sessions...")
print("   (Watch for the agent to use search_long_term_memory tool)\n")

response = await agent_q2.run(
    "What patients am I treating? What are their symptoms, treatments, and clinical guidelines?",
    memory=memory_q2,
)

print("\n🧠 Q2 Patient Recall:")
print(response)
print("\n✅ Expected: PATIENT-001, diabetes symptoms, metformin treatment, lifestyle guidelines")

In [ ]:
# 치료 protocol 문서화
response = await agent_q2.run(
    "Document treatment protocol: protocol type 'Type 2 Diabetes Management', "
    "indication 'newly diagnosed T2DM with HbA1c 8.2%', "
    "effectiveness 'HbA1c reduced to 7.1% after 3 months', "
    "side effects 'mild GI upset initially, resolved with food intake'.",
    memory=memory_q2,
)
print("🌍 Q2 Treatment Protocol:", response)

# 반응에 따라 임상 guideline 업데이트
response = await agent_q2.run(
    "Update clinical guideline for 'PATIENT-001': guideline type 'Medication Adjustment', "
    "recommendation 'continue metformin 500mg BID, add glipizide 5mg daily for additional glycemic control', evidence level 'high'.",
    memory=memory_q2,
)
print("⚖️ Q2 Guideline Update:", response)

In [ ]:
# Q2 임상 metric 추적
response = await agent_q2.run(
    "Track clinical metrics for 'PATIENT-001': metric type 'HbA1c Improvement', value '8.2% to 7.1%', "
    "patient_id 'PATIENT-001', quarter 'Q2 2024'.",
    memory=memory_q2,
)
print("📈 Q2 Clinical Metrics:", response)

# 치료 비교 테스트
response = await agent_q2.run(
    "How did PATIENT-001's treatment progress from Q1 to Q2? Compare initial vs current approach.",
    memory=memory_q2,
)
print("📊 Q2 Treatment Analysis:")
print(response)
print("\n✅ Expected: Q1 metformin monotherapy → Q2 combination therapy, HbA1c improvement")

## 7단계: Q3 의료 Session - 합병증 관리

합병증 관리 단계로 진행하고 새 환자를 접수합니다.

In [ ]:
# === Q3 의료 SESSION 시작 ===
print("\n🗓️ === Q3: COMPLICATION MANAGEMENT AND NEW PATIENT ===")

agent_q3, memory_q3 = create_medical_session("q3")

# 치료 결과 기록
response = await agent_q3.run(
    "Log treatment outcome for 'PATIENT-001' with treatment 'Diabetes Management Protocol', "
    "outcome 'Target HbA1c achieved (6.8%), weight loss 15 lbs, BP controlled', "
    "follow_up_needed 'quarterly HbA1c monitoring, annual eye exam, continue current regimen'.",
    memory=memory_q3,
)
print("📅 Q3 Treatment Outcome:", response)

# 새 환자 평가 시작
response = await agent_q3.run(
    "Assess patient symptoms for 'PATIENT-002': symptoms 'chest pain, shortness of breath, palpitations', "
    "severity 'High', duration '2 days'.",
    memory=memory_q3,
)
print("💭 Q3 New Patient Assessment:", response)

In [ ]:
# 종합적인 의료 기록 회상 테스트
response = await agent_q3.run(
    "What is the complete medical care history? Include all patients, treatments, protocols, and outcomes.",
    memory=memory_q3,
)
print("📋 Q3 Complete History:")
print(response)
print("\n✅ Expected: PATIENT-001 diabetes journey → PATIENT-002 cardiac assessment, protocol evolution")

## 8단계: Q4 의료 Session - 연말 검토 및 계획

Semantic search와 연간 의료 분석을 테스트합니다.

In [ ]:
# === Q4 의료 SESSION 시작 ===
print("\n🗓️ === Q4: YEAR-END REVIEW AND PLANNING ===")

agent_q4, memory_q4 = create_medical_session("q4")

# 연간 의료 metric 추적
response = await agent_q4.run(
    "Track clinical metrics: metric type '2024 Annual Performance', value 'Patients treated: 2, Diabetes control achieved: 100%, Complications prevented: 1', "
    "patient_id 'ANNUAL-SUMMARY', quarter '2024 Annual'.",
    memory=memory_q4,
)
print("📈 Q4 Annual Metrics:", response)

# 치료 protocol 상관관계 테스트
response = await agent_q4.run(
    "What treatment protocols have I documented this year? How effective were they?",
    memory=memory_q4,
)
print("🌍 Q4 Protocol Effectiveness Analysis:")
print(response)
print("\n✅ Expected: Diabetes management protocol → successful HbA1c control")

In [ ]:
# 유사한 의료 접근 방식의 semantic search 테스트
response = await agent_q4.run(
    "What clinical guidelines have I used? Which were most effective based on patient outcomes?",
    memory=memory_q4,
)
print("⚖️ Q4 Guideline Effectiveness Analysis:")
print(response)
print("\n✅ Expected: Diabetes management + lifestyle modification = successful outcomes")

## 9단계: 2년 차 Q1 Session - 다년간 의료 관점

장기 의료 지식과 업무 변화를 테스트합니다.

In [ ]:
# === 2년 차 Q1 의료 SESSION 시작 ===
print("\n🗓️ === YEAR 2 Q1: MULTI-YEAR MEDICAL PERSPECTIVE ===")

agent_y2q1, memory_y2q1 = create_medical_session("year2-q1")

# 다년간 의료 업무 분석
response = await agent_y2q1.run(
    "Analyze my medical practice evolution: How have my patients and treatments developed over the past year? "
    "What were the key clinical decisions and their outcomes?",
    memory=memory_y2q1,
)
print("📊 Year 2 Q1 Practice Analysis:")
print(response)
print("\n✅ Expected: PATIENT-001 → PATIENT-002 progression, protocol refinement, outcome improvement")

In [ ]:
# 임상 protocol 변화 추적 테스트
response = await agent_y2q1.run(
    "How have my clinical protocols and guidelines evolved? What treatments have I applied and why?",
    memory=memory_y2q1,
)
print("💭 Year 2 Q1 Protocol Evolution:")
print(response)
print("\n✅ Expected: Started with diabetes protocols → expanded to cardiac care")

## 10단계: 최종 의료 업무 평가

장기 의료 분석 기능을 종합적으로 테스트합니다.

In [ ]:
# 최종 종합 의료 업무 질의
response = await agent_y2q1.run(
    "Provide my complete medical practice portfolio: all patients with their clinical journeys, "
    "treatment effectiveness, protocol applications, guideline utilization, and patient outcomes. "
    "Include lessons learned and best practices developed.",
    memory=memory_y2q1,
)
print("💼 Complete Medical Practice Portfolio:")
print(response)
print("\n✅ Expected: Full patient portfolio with treatment evolution, protocol effectiveness, and outcome analysis")

## 🧪 자동 테스트 검증
이 셀을 실행하여 메모리 통합이 올바르게 작동하는지 검증합니다.

In [ ]:
# Validation function을 inline으로 정의
class TestValidator:
    def __init__(self):
        self.results = {}

    def validate_memory_recall(self, response):
        """에이전트가 세션 앞부분의 정보를 기억하는지 확인합니다."""
        # "I don't know"만 반환한 것이 아닌 실질적인 응답인지 확인
        has_content = len(response) > 50
        # Memory indicator 확인
        has_memory_indicators = any(
            word in response.lower()
            for word in [
                "earlier",
                "mentioned",
                "discussed",
                "previously",
                "you",
                "we",
                "our",
            ]
        )
        return "✅ PASS" if (has_content and has_memory_indicators) else "❌ FAIL"

    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인합니다."""
        has_memory_content = len(response) > 100 and any(
            word in response.lower()
            for word in [
                "previous",
                "earlier",
                "mentioned",
                "discussed",
                "before",
                "already",
            ]
        )
        return "✅ PASS" if has_memory_content else "❌ FAIL"

    def validate_cross_reference(self, response):
        """에이전트가 현재 질의를 이전 컨텍스트와 연결할 수 있는지 확인합니다."""
        # 연결 표현 확인
        connecting_words = [
            "relate",
            "connection",
            "previous",
            "earlier",
            "discussed",
            "mentioned",
            "context",
            "based on",
            "as we",
            "as i",
        ]
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "✅ PASS" if (has_connection and has_substance) else "❌ FAIL"

    def run_validation_summary(self, test_results):
        print("🧪 COMPREHENSIVE TEST VALIDATION SUMMARY")
        print("=" * 60)

        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "PASS" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0

        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")

        print("=" * 60)
        print(f"📊 Overall Pass Rate: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")

        if pass_rate >= 80:
            print("✅ EXCELLENT: Memory integration working correctly!")
        elif pass_rate >= 60:
            print("⚠️  GOOD: Most memory features working, some issues to investigate")
        else:
            print("❌ NEEDS ATTENTION: Memory integration has significant issues")

        return pass_rate


validator = TestValidator()
print("✅ Validation functions loaded!")

In [ ]:
# 모든 validation test 실행
test_results = {}

# 테스트 1: Memory 회상 - 에이전트가 논의 내용을 기억하는가?
response1 = await agent_y2q1.run("What have we discussed so far in this session?", memory=memory_y2q1)
test_results["Memory Recall"] = validator.validate_memory_recall(str(response1))
print(f"Response 1 length: {len(str(response1))} chars")

# 테스트 2: Session memory - 에이전트가 맥락을 유지하는가?
response2 = await agent_y2q1.run("What did we talk about earlier?", memory=memory_y2q1)
test_results["Session Memory"] = validator.validate_session_memory(str(response2))
print(f"Response 2 length: {len(str(response2))} chars")

# 테스트 3: 상호 참조 기능 - 이전 맥락과 연결할 수 있는가?
response3 = await agent_y2q1.run("How does this relate to what we discussed before?", memory=memory_y2q1)
test_results["Cross Reference"] = validator.validate_cross_reference(str(response3))
print(f"Response 3 length: {len(str(response3))} chars")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 Notebook에서는 다음 내용을 살펴봤습니다.

✅ **장기 메모리 통합**: LlamaIndex와 AgentCore Memory를 사용하여 session 간 의료 분석 구현

✅ **환자 진료 추적**: 여러 분기에 걸친 환자 상태 변화와 치료 발전 추적

✅ **임상 Intelligence**: 치료 protocol과 적용 사례를 semantic 방식으로 검색

✅ **의료 Protocol 변화**: 초기 평가에서 근거 기반 접근 방식으로 자연스럽게 발전

✅ **치료 효과**: 임상 결정과 환자 결과를 상세하게 추적

✅ **우수한 의료 업무**: 시간에 따른 종합적인 환자 관리 및 진료 최적화

Medical Knowledge Assistant는 장기 메모리를 통해 전체 환자 기록을 유지하고 장기 진료 기간 전반에서 정교한 임상 지식 검색을 지원하며 시간이 지날수록 더 똑똑해지는 지속적인 의료 partner로 발전할 수 있음을 보여 줍니다.

## 정리

이 Notebook에서 사용한 resource를 정리하도록 memory를 삭제하겠습니다.

**참고**: Memory를 영구 삭제하려는 경우에만 실행하세요. memory_id 변수에는 이 Notebook의 앞부분에서 생성한 memory의 ID가 있어야 합니다.

In [ ]:
# AgentCore Memory resource 정리
try:
    from bedrock_agentcore.memory import MemoryClient

    client = MemoryClient(region_name=region)
    client.delete_memory(memory_id)
    print(f"✅ Successfully deleted memory: {memory_id}")

except NameError as e:
    print(f"⚠️  Variable not defined: {e}")
    print("Run the notebook from the beginning or set variables manually:")
    print("# memory_id = 'your-memory-id-here'")
    print("# region = 'us-east-1'")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")